In [1]:
import pandas as pd
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, r2_score, precision_score, recall_score
import numpy as np

In [2]:
# 1. Load Data
print("Loading engineered features...")
df = pd.read_csv('./data/engineered_features_all.csv', parse_dates=['Timestamp'])
df = df.sort_values(['Cell_ID', 'Timestamp'])

Loading engineered features...


In [5]:
# 2. Strict Train/Test Split (with 6-hour gap to prevent target leakage)
print("Splitting data into Train and Test sets...")

# FIX: Calculate the 80th percentile of the actual time span
split_date = df['Timestamp'].quantile(0.8)

# Adding a 6-hour gap ensures the model's training targets don't overlap with the test inputs
gap_date = split_date + pd.Timedelta(hours=6)

train_df = df[df['Timestamp'] <= split_date]
test_df = df[df['Timestamp'] > gap_date]

features = [
    '4G data volume DL', '4G RB utilization', '4G active users DL', '4G CQI rank 1', 
    'hour', 'day_of_week', 'vol_lag_1', 'rb_lag_1', 'users_lag_1'
]
rb_targets = [f'rb_target_{h}h' for h in range(1, 7)]

X_train, y_train_rb = train_df[features], train_df[rb_targets]
X_test, y_test_rb = test_df[features], test_df[rb_targets]

print(f"Training on {len(X_train)} intervals (Dates <= {split_date.date()})")
print(f"Testing on {len(X_test)} intervals (Dates > {gap_date.date()})")

Splitting data into Train and Test sets...
Training on 411618 intervals (Dates <= 2025-04-26)
Testing on 99975 intervals (Dates > 2025-04-26)


In [6]:
# 3. Train Model (Focusing on RB Utilization for the Business Case)
print("\nTraining XGBoost on RB Utilization...")
model = MultiOutputRegressor(
    xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, tree_method='hist', random_state=42)
)
model.fit(X_train, y_train_rb)
y_pred_rb = model.predict(X_test)


Training XGBoost on RB Utilization...


In [7]:
# 4. Rigorous Evaluation (1-Hour Ahead)
target_1h = y_test_rb['rb_target_1h']
pred_1h = y_pred_rb[:, 0]

# Baselines
persistance_1h = test_df['4G RB utilization'] # Forecast = Value right now
seasonal_1h = test_df['baseline_rb_24h']      # Forecast = Value yesterday

def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

print("\n--- 1-Hour Ahead RB Utilization Evaluation ---")
print(f"XGBoost MAE:         {mean_absolute_error(target_1h, pred_1h):.2f}%")
print(f"Persistence MAE:     {mean_absolute_error(target_1h, persistance_1h):.2f}%")
print(f"Seasonal Naive MAE:  {mean_absolute_error(target_1h, seasonal_1h):.2f}%")

print(f"\nXGBoost WAPE:        {wape(target_1h, pred_1h):.2%}")
print(f"XGBoost R²:          {r2_score(target_1h, pred_1h):.2f}")


--- 1-Hour Ahead RB Utilization Evaluation ---
XGBoost MAE:         4.91%
Persistence MAE:     5.59%
Seasonal Naive MAE:  7.45%

XGBoost WAPE:        30.38%
XGBoost R²:          0.73


In [8]:
# 5. Business Logic Validation
print("\n--- Business Logic Classification (1-Hour Ahead) ---")
# Sleep Mode (<10% utilization)
true_sleep = target_1h < 10
pred_sleep = pred_1h < 10
print("Sleep Mode Decision (<10% RB):")
print(f"  Precision: {precision_score(true_sleep, pred_sleep, zero_division=0):.2f}")
print(f"  Recall:    {recall_score(true_sleep, pred_sleep, zero_division=0):.2f}")

# Traffic Steering (>85% utilization)
true_steer = target_1h > 85
pred_steer = pred_1h > 85
if true_steer.sum() > 0:
    print("\nTraffic Steering Decision (>85% RB):")
    print(f"  Precision: {precision_score(true_steer, pred_steer, zero_division=0):.2f}")
    print(f"  Recall:    {recall_score(true_steer, pred_steer, zero_division=0):.2f}")
else:
    print("\nTraffic Steering Decision (>85% RB): No instances of >85% congestion in the test set.")


--- Business Logic Classification (1-Hour Ahead) ---
Sleep Mode Decision (<10% RB):
  Precision: 0.85
  Recall:    0.73

Traffic Steering Decision (>85% RB): No instances of >85% congestion in the test set.
